# A/B Testing — Live In-Class Experiment
### ECON 4370 / BANA 4373 — Applied Data Tools | Spring 2026

---

**Scenario**: An online retailer built two versions of a promotional landing page.

| Group | Page | n |
|---|---|---|
| **Control (A)** | Existing page | 400 |
| **Treatment (B)** | Redesigned page | 400 |

**Outcome**: did the visitor make a purchase? `y = 1` (yes) or `y = 0` (no).

**Baseline conversion rate**: 20% (roughly — we will estimate it from the data).

---

### How to use this notebook
- Every cell is already written — just **run it in order** (Shift+Enter)
- 💬 cells ask you to **discuss** something with your neighbour or answer a question
- ✏️ cells ask you to **write a short answer** in the markdown cell below the code
- You do not need to write any code today — focus on understanding the output


## 0. Setup — run this first

This loads the libraries we will use throughout.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

try:
    import statsmodels.formula.api as smf
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print('Libraries loaded OK.')

## 1. Load the experiment data

The cell below generates a reproducible random experiment using a fixed seed.
Think of it as downloading the results file from the data team.

Each row is one visitor:
- `unit` — visitor ID
- `treat` — 1 if assigned to the new page, 0 if assigned to the old page
- `y` — 1 if they purchased, 0 if they did not
- `group` — same as treat, but labelled Control / Treatment


In [ ]:
# ── Experiment generator ──────────────────────────────────────────────────
def make_experiment(seed, n_per_group=400, baseline_rate=0.20, true_lift=0.03):
    rng = np.random.default_rng(seed)
    treat = np.r_[np.zeros(n_per_group, dtype=int), np.ones(n_per_group, dtype=int)]
    p = np.where(treat == 1, baseline_rate + true_lift, baseline_rate)
    y = rng.binomial(1, p)
    df = pd.DataFrame({'unit': np.arange(len(treat)), 'treat': treat, 'y': y})
    df['group'] = np.where(df['treat'] == 1, 'Treatment', 'Control')
    return df

# ── MDE approximation ────────────────────────────────────────────────────────
def approximate_mde(n_per_group, sigma=0.45, alpha=0.05, power=0.80):
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_power = stats.norm.ppf(power)
    return (z_alpha + z_power) * sigma * np.sqrt(2 / n_per_group)

# ── Repeated experiments simulator ──────────────────────────────────────────
def run_many_experiments(n_sims=2000, n_per_group=400, baseline_rate=0.20,
                         true_lift=0.03, seed=2026):
    rng = np.random.default_rng(seed)
    lifts = np.empty(n_sims)
    for s in range(n_sims):
        c = rng.binomial(1, baseline_rate, n_per_group)
        t = rng.binomial(1, baseline_rate + true_lift, n_per_group)
        lifts[s] = t.mean() - c.mean()
    return lifts

# ── Peeking simulator ────────────────────────────────────────────────────────
def peeking_false_positive_rate(n_total=1000, looks=10, sims=1500, seed=2026):
    rng = np.random.default_rng(seed)
    checkpoints = np.unique(np.linspace(50, n_total, looks, dtype=int))
    false_positives = 0
    for _ in range(sims):
        data = rng.binomial(1, 0.20, n_total * 2)
        c_all, t_all = data[:n_total], data[n_total:]
        for cp in checkpoints:
            c, t = c_all[:cp], t_all[:cp]
            if len(c) < 2:
                continue
            se = np.sqrt(t.mean()*(1-t.mean())/len(t) + c.mean()*(1-c.mean())/len(c))
            if se == 0:
                continue
            z = (t.mean() - c.mean()) / se
            if 2 * (1 - stats.norm.cdf(abs(z))) < 0.05:
                false_positives += 1
                break
    return false_positives / sims

# ── Parameters ───────────────────────────────────────────────────────────────
SEED         = 2026
N_PER_GROUP  = 400
BASELINE     = 0.20
ALPHA        = 0.05

df = make_experiment(seed=SEED, n_per_group=N_PER_GROUP, baseline_rate=BASELINE)
print(f'Dataset loaded: {len(df)} rows')
df.head(8)

---
## ✏️ Before we look at results — write your prior

The baseline conversion rate is **20%**. The treatment is a redesigned landing page.

**Question**: How many extra percentage points do you expect the new page to convert?
For example, if you think it will go from 20% to 23%, write **3**.

> **My guess**: _____ percentage points

We will return to this at the end. Do not change it after you see the results!


---
## 2. Summarize the groups

For each group we compute:
- **users**: total number of visitors in the group
- **conversions**: number who purchased (`y = 1`)
- **conversion_rate**: fraction who purchased = conversions / users

`groupby('group')` splits the data by group, then `.agg()` computes
the three statistics simultaneously for each group.


In [ ]:
group_summary = (
    df.groupby('group')
      .agg(
          users          = ('y', 'size'),
          conversions    = ('y', 'sum'),
          conversion_rate= ('y', 'mean')
      )
      .reset_index()
)
group_summary

### 💬 Discussion

1. Which group has the higher conversion rate?
2. The difference looks positive — does that mean the new page works?
   What else do we need to check?

> *Your thoughts:*


---
### 2a. Balance check — did randomization work?

Before we trust the result, we verify that the two groups were comparable
**before** treatment. If randomization worked, pre-treatment characteristics
should be statistically indistinguishable.

We only have one pre-treatment variable here: `unit` (the visitor ID).
In a real dataset you would check age, prior purchases, device type, etc.

The **standardized mean difference (SMD)** measures imbalance:

$$d = \frac{\bar{X}_{\text{treatment}} - \bar{X}_{\text{control}}}
{\sqrt{(s^2_T + s^2_C)/2}}$$

Rule of thumb: $|d| < 0.1$ means the groups are well-balanced.


In [ ]:
mean_unit_c   = df.loc[df.treat == 0, 'unit'].mean()
mean_unit_t   = df.loc[df.treat == 1, 'unit'].mean()
std_unit_c    = df.loc[df.treat == 0, 'unit'].std()
std_unit_t    = df.loc[df.treat == 1, 'unit'].std()
pooled_std    = np.sqrt((std_unit_c**2 + std_unit_t**2) / 2)
smd           = (mean_unit_t - mean_unit_c) / pooled_std

print(f'Mean unit ID  — Control:   {mean_unit_c:.1f}')
print(f'Mean unit ID  — Treatment: {mean_unit_t:.1f}')
print(f'Standardized difference:   {smd:.3f}')
print('(Unit ID is sequential, so large SMD is expected — not a real covariate)')

### 💬 Discussion

The SMD for `unit` is very large — but that is expected and harmless. Why?
The unit IDs run 0–399 for control and 400–799 for treatment by construction.
A visitor ID assigned before randomization is not a real covariate.

In a real experiment, what covariate would you most want to check for balance?

> *Your thoughts:*


---
## 3. Estimate the treatment effect

Under randomization, the **ATE estimator** is simply the difference in group means:

$$\hat{\tau} = \bar{Y}_{\text{treatment}} - \bar{Y}_{\text{control}}$$

We also compute:

| Quantity | Formula | Meaning |
|---|---|---|
| **SE** | $\sqrt{\frac{p_T(1-p_T)}{n_T} + \frac{p_C(1-p_C)}{n_C}}$ | How noisy is our estimate? |
| **95% CI** | $\hat{\tau} \pm 1.96 \times \text{SE}$ | Plausible range for the true effect |
| **z-stat** | $\hat{\tau} / \text{SE}$ | How many SEs away from zero? |
| **p-value** | $2 \times P(Z > |z|)$ | How surprising under H0? |


In [ ]:
# ── Group statistics ─────────────────────────────────────────────────────
mean_c = df.loc[df.treat == 0, 'y'].mean()
mean_t = df.loc[df.treat == 1, 'y'].mean()
n_c    = (df.treat == 0).sum()
n_t    = (df.treat == 1).sum()

# ── ATE estimate ─────────────────────────────────────────────────────────────
lift   = mean_t - mean_c

# ── Standard error of the difference in proportions ─────────────────────────
se     = np.sqrt(mean_t*(1 - mean_t)/n_t  +  mean_c*(1 - mean_c)/n_c)

# ── 95% Confidence interval ──────────────────────────────────────────────────
z_crit  = stats.norm.ppf(1 - ALPHA / 2)   # = 1.96
ci_low  = lift - z_crit * se
ci_high = lift + z_crit * se

# ── z-statistic and two-tailed p-value ──────────────────────────────────────
z_stat  = lift / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print(f'Control conversion rate :  {mean_c:.3%}  (n={n_c})')
print(f'Treatment conversion rate: {mean_t:.3%}  (n={n_t})')
print(f'Estimated lift (tau_hat) : {lift:.3%}')
print(f'Standard error           : {se:.3%}')
print(f'95% CI                   : [{ci_low:.3%}, {ci_high:.3%}]')
print(f'z-statistic              : {z_stat:.3f}')
print(f'p-value                  : {p_value:.4f}')
print()
print(f'Decision: {"Reject H0 — significant" if p_value < ALPHA else "Fail to reject H0 — not significant"} at alpha={ALPHA}')

### 💬 Discussion

1. **Is the result statistically significant at the 5% level?** How do you know?
   (Three equivalent ways to answer: p-value, z-statistic, or confidence interval.)

2. **Is the confidence interval entirely above zero?** What does that tell you?

3. **Would you recommend shipping the new page?** What additional information
   would you want before making that decision?

4. **Compare to your prior**: was your guess close to the estimate?

> *Your thoughts:*


---
## 4. The OLS — t-test connection

This is one of the most important results in statistics for practitioners.

**Claim**: running the regression `y ~ treat` with OLS gives *exactly* the same
estimate as the difference in means — same coefficient, same standard error, same p-value.

### Why? The algebra

OLS fits the model $y_i = \alpha + \tau \cdot \text{treat}_i + \varepsilon_i$.
When `treat` is binary (0 or 1), the OLS solution is:

$$\hat{\alpha} = \bar{Y}_{\text{control}} \qquad \hat{\tau} = \bar{Y}_{\text{treatment}} - \bar{Y}_{\text{control}}$$

**Proof sketch**: OLS minimises $\sum(y_i - \alpha - \tau D_i)^2$.
The first-order conditions give:
- $\partial / \partial\alpha$: $\hat\alpha = \bar{Y}_C$ (intercept = control mean)
- $\partial / \partial\tau$: $\hat\tau = \bar{Y}_T - \bar{Y}_C$ (slope = difference in means)

The cell below verifies this numerically.

### Why does this matter?
Because OLS immediately generalises: add covariates $X_i$ to the model and
you reduce the residual variance, shrinking the SE and improving precision —
without changing the expected value of $\hat\tau$ (under randomization).
This is called **ANCOVA** and is the standard in industry A/B analysis.


In [ ]:
# ── Step 1: Run the OLS regression ──────────────────────────────────────────
if HAS_STATSMODELS:
    reg = smf.ols('y ~ treat', data=df).fit()

    # ── Step 2: Extract the pieces ───────────────────────────────────────────
    intercept   = reg.params['Intercept']
    treat_coef  = reg.params['treat']

    print('OLS regression:  y_i = alpha + tau * treat_i + epsilon_i')
    print('='*55)
    print(f'  Intercept (alpha) = {intercept:.6f}')
    print(f'  treat coef (tau)  = {treat_coef:.6f}')
    print()
    print('Direct computation:')
    print(f'  Control mean      = {mean_c:.6f}   <- should equal intercept')
    print(f'  Treatment mean    = {mean_t:.6f}')
    print(f'  Difference        = {lift:.6f}   <- should equal treat coef')
    print()
    print(f'  Intercept == Control mean?  {abs(intercept - mean_c) < 1e-10}')
    print(f'  Treat coef == Lift?         {abs(treat_coef - lift) < 1e-10}')
    print()
    print('Full regression table:')
    print(reg.summary().tables[1])
else:
    print('statsmodels not available.')

### 💬 Discussion

1. The intercept from OLS equals the control group mean. Why does that make sense
   geometrically? (Think: what does the regression line look like when X is 0/1?)

2. The standard errors in the OLS table should also match the SE from Section 3.
   Check: do they?

3. If you added `baseline_purchases` as a covariate, would you expect `tau` to change?
   Would you expect the SE to change? Which direction?

> *Your thoughts:*


---
## 5. Visualize the result

Two panels:
- **Left**: conversion rate per group with 95% CI error bars around each group mean
- **Right**: 95% CI for the *difference* (the thing we actually care about)

> Note: the left panel error bars and the right panel CI are **different quantities**.
> The left bars show uncertainty around each group mean separately.
> The right panel shows uncertainty around their difference.


In [ ]:
z_crit = stats.norm.ppf(1 - ALPHA / 2)
ci_half_c = z_crit * np.sqrt(mean_c * (1 - mean_c) / n_c)
ci_half_t = z_crit * np.sqrt(mean_t * (1 - mean_t) / n_t)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# ── Left: bar chart ───────────────────────────────────────────────────────────
ax = axes[0]
ax.bar(['Control', 'Treatment'], [mean_c, mean_t],
       yerr=[ci_half_c, ci_half_t],
       color=['#007681', '#C9A454'], width=0.5,
       capsize=6, error_kw={'linewidth': 1.5, 'ecolor': '#444'})
ax.set_ylabel('Conversion rate')
ax.set_title('Conversion rate by group\n(error bars = 95% CI)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_ylim(0, max(mean_c, mean_t) * 1.6)

# ── Right: CI for the difference ─────────────────────────────────────────────
ax2 = axes[1]
ax2.axvline(0, color='gray', linestyle='--', linewidth=1, label='No effect')
ax2.errorbar(lift, 0, xerr=z_crit * se, fmt='o',
             color='#00264A', markersize=8, linewidth=2,
             label=f'Lift = {lift:.2%}')
ax2.set_xlabel('Treatment effect (percentage points)')
ax2.set_title('95% CI for the difference\n(tau_hat +/- 1.96 * SE)')
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.set_yticks([])
ax2.legend()

plt.tight_layout()
plt.show()
print(f'Left panel: error bars show uncertainty around EACH group mean separately.')
print(f'Right panel: CI shows uncertainty around the DIFFERENCE. Zero {"excluded" if ci_low > 0 else "included"}.')

### 💬 Discussion

1. On the right panel: does the CI include zero or exclude zero?
   What does that mean for our hypothesis test?

2. Two groups can have overlapping error bars on the left panel and still have
   a significant difference on the right panel (or vice versa). Why? Which
   panel is the correct one to use for the hypothesis test?

> *Your thoughts:*


---
## 6. Sampling noise — what if we ran this 2,000 times?

Our experiment produced **one** estimate. But if we could rerun the same experiment
on a fresh random sample, we would get a different estimate every time — even though
the true effect is always the same.

The cell below simulates 2,000 independent experiments all with the same true effect
and plots the distribution of estimated lifts. The gold vertical line is the true lift.
The red dashed line is *our* single estimate from today.


In [ ]:
lifts_sim = run_many_experiments(
    n_sims=2000, n_per_group=N_PER_GROUP,
    baseline_rate=BASELINE, seed=SEED
)

plt.figure(figsize=(9, 4))
plt.hist(lifts_sim * 100, bins=45, edgecolor='white', color='steelblue', alpha=0.8)
plt.axvline(0.03 * 100, color='#C9A454', linewidth=2.5, label='True lift = 3 pp')
plt.axvline(lift  * 100, color='#C02C2C', linewidth=2,   linestyle='--',
            label=f'Our estimate = {lift:.2%}')
plt.xlabel('Estimated lift (percentage points)')
plt.ylabel('Count of simulated experiments')
plt.title('What if we ran this experiment 2,000 times?')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean of simulated lifts : {lifts_sim.mean():.3%}  (should be close to true 3.00%)')
print(f'Std  of simulated lifts : {lifts_sim.std():.3%}  (should be close to SE={se:.3%})')
print(f'Our single estimate     : {lift:.3%}')
print(f'Fraction of sims <= 0   : {(lifts_sim <= 0).mean():.1%}  (would wrongly conclude no effect)')

### 💬 Discussion

1. **What does the spread (width) of this distribution represent?**
   What determines how wide or narrow it is?

2. **Compare the standard deviation of the simulated lifts to the SE from Section 3.**
   Are they close? Why should they be?

3. **Look at our single estimate (red line) vs the true lift (gold line).**
   Our estimate is much higher than the truth. Is that a problem with our experiment?
   What does it tell you about how to interpret a single experimental result?

4. **What fraction of simulated experiments gave a lift of zero or below,**
   even though the true lift is positive? What does that fraction represent?

> *Your thoughts:*


---
## 7. Power and minimum detectable effects

**Power** is the probability of detecting an effect when one truly exists.
You must decide your required power *before* running the experiment.

The **minimum detectable effect (MDE)** is the smallest true lift your
experiment can reliably detect (at your chosen power level) given its sample size:

$$\text{MDE} = (z_{1-\alpha/2} + z_{1-\beta}) \cdot \sigma \cdot \sqrt{\frac{2}{n}}$$

| Symbol | Meaning | Default |
|---|---|---|
| $z_{1-\alpha/2}$ | Critical value for significance | 1.96 (for $\alpha=5\%$) |
| $z_{1-\beta}$ | Critical value for power | 0.84 (for $80\%$ power) |
| $\sigma$ | Std dev of the outcome | ~0.45 for binary at 20% |
| $n$ | Sample size **per group** | 400 today |

**If the true effect is smaller than the MDE, your experiment will usually miss it.**


In [ ]:
mde_today = approximate_mde(N_PER_GROUP)
print(f'MDE for n = {N_PER_GROUP} per group : {mde_today:.2%}')
print(f'True lift               : 3.00%  (revealed at end of class)')
print(f'Our estimate            : {lift:.2%}')
print()
powered = 0.03 >= mde_today
print(f'Was the experiment well-powered to detect the TRUE 3pp effect?')
print(f'  MDE = {mde_today:.2%},  True lift = 3.00%')
print(f'  Answer: {"YES" if powered else "NO — underpowered"}')

In [ ]:
sample_sizes = np.arange(100, 5100, 100)
mde_values   = np.array([approximate_mde(n) for n in sample_sizes])

plt.figure(figsize=(9, 4))
plt.plot(sample_sizes, mde_values * 100, color='#007681', linewidth=2)
plt.axhline(3.0,          color='#C9A454', linestyle='--', linewidth=1.5,
            label='True lift = 3 pp')
plt.axvline(N_PER_GROUP,  color='#C02C2C', linestyle=':',  linewidth=1.5,
            label=f"Today's n = {N_PER_GROUP}")
plt.xlabel('Sample size per group')
plt.ylabel('Approx. MDE (percentage points)')
plt.title('Larger samples detect smaller effects')
plt.legend()
plt.tight_layout()
plt.show()

# Find required n to detect 3pp
n_for_3pp = sample_sizes[mde_values <= 0.03][0] if any(mde_values <= 0.03) else '>5000'
print(f'Minimum n per group to detect 3pp effect: ~{n_for_3pp}')

In [ ]:
print(f'{"n per group":>12}  {"MDE":>10}  {"Detects 3pp?"}' )
print('-' * 45)
for n in [200, 400, 1000, 2500, 5000]:
    mde = approximate_mde(n)
    flag = 'YES' if 0.03 >= mde else 'No (underpowered)'
    print(f'{n:>12}  {mde:>10.2%}  {flag}')

### 💬 Discussion

1. **Was today's experiment well-powered to detect the true 3pp effect?**
   Explain using the MDE table.

2. **We got a significant result anyway.** Given that we were underpowered,
   what does that tell you about our estimate of 8.75pp vs the true 3pp?
   (This is called **Type M error** — magnitude error.)

3. **The MDE shrinks with larger n, but not linearly.**
   If you double the sample size, by what factor does the MDE shrink?
   (Hint: look at the formula — $n$ appears under a square root.)

4. **Business question**: if the company only cares about effects larger than 5pp,
   what sample size per group would they need?
   Use the chart or table to read off the answer.

> *Your thoughts:*


---
## 8. The peeking problem

**Peeking** = checking the results before the pre-specified end of the experiment
and stopping early whenever $p < 0.05$.

This is one of the most common bad practices in industry A/B testing.
The simulation below runs experiments on **pure noise** (true effect = 0)
and counts how often we falsely declare significance, as a function of
how many times we check.


In [ ]:
looks_grid = np.arange(1, 21)
fp_rates   = [
    peeking_false_positive_rate(looks=int(k), sims=1000, seed=SEED + int(k))
    for k in looks_grid
]

plt.figure(figsize=(9, 4))
plt.plot(looks_grid, np.array(fp_rates) * 100, marker='o', markersize=5,
         color='#00264A', linewidth=2)
plt.axhline(5, color='#C02C2C', linestyle='--', linewidth=1.5,
            label='Nominal 5% level')
plt.fill_between(looks_grid, 5, np.array(fp_rates)*100,
                 where=np.array(fp_rates)*100 > 5,
                 alpha=0.15, color='#C02C2C', label='Excess false positives')
plt.xlabel('Number of times you peek at the data')
plt.ylabel('False positive rate (%)')
plt.title('Peeking inflates Type I error')
plt.legend()
plt.tight_layout()
plt.show()

fp_at_10 = fp_rates[9] * 100
print(f'False positive rate at 10 peeks: {fp_at_10:.1f}%  (should be 5% if done correctly)')

### 💬 Discussion

1. **At 10 peeks, what is the false positive rate?**
   What does that mean in plain English for a business that runs 100 tests per year?

2. **A colleague says**: *'I checked after 500 users and it wasn't significant,
   so I kept going. I checked again at 800 and it was significant, so I stopped.'*
   How many peeks did they make? What is their real false positive rate?

3. **Name one method** that allows you to look at data during an experiment
   without inflating the false positive rate.

4. **Connection to multiple testing**: the peeking problem is mathematically
   identical to running multiple independent tests and reporting the best one.
   Can you explain why?

> *Your thoughts:*


---
## 9. Non-compliance: ITT and LATE

In a perfect experiment everyone assigned to treatment actually receives it.
In the real world, that rarely happens.

**Our scenario**: imagine that 30% of treatment-assigned visitors had a browser
caching issue and saw the old page instead of the new one.
We now have two different variables:

| Variable | Meaning | Randomized? |
|---|---|---|
| `treat` | Was the visitor **assigned** to the new page? | ✅ Yes |
| `received` | Did the visitor **actually see** the new page? | ❌ No — affected by the bug |

This creates two estimands:

| Estimand | Uses | Answers |
|---|---|---|
| **ITT** (Intention-to-Treat) | Assignment (`treat`) | What is the effect of *offering* the treatment? |
| **LATE** (Local Average Treatment Effect) | Both | What is the effect for those who *actually complied*? |


In [ ]:
rng_nc           = np.random.default_rng(SEED + 99)
COMPLIANCE_RATE  = 0.70

df_nc = df.copy()
treatment_idx     = df_nc[df_nc.treat == 1].index
non_complier_mask = rng_nc.random(len(treatment_idx)) > COMPLIANCE_RATE
non_complier_idx  = treatment_idx[non_complier_mask]

df_nc['received'] = df_nc['treat'].copy()         # start: everyone receives as assigned
df_nc.loc[non_complier_idx, 'received'] = 0       # 30% of treatment group did NOT receive it

print('Assignment vs actual receipt:')
print(df_nc.groupby(['treat','received']).size().rename('count').reset_index().to_string(index=False))
print()
actual_cr = df_nc.loc[df_nc.treat==1,'received'].mean()
print(f'Actual compliance rate among assigned-treatment: {actual_cr:.1%}')

### 9a. Intention-to-Treat (ITT)

The ITT compares based on **assignment** (`treat`), regardless of actual receipt.
Because `treat` was randomized, the ITT is always unbiased — but it understates
the per-complier effect because 30% of the 'treatment' group got nothing.

$$\text{ITT} = E[Y \mid Z=1] - E[Y \mid Z=0]$$

where $Z$ = assignment indicator.


In [ ]:
mean_assigned_t  = df_nc.loc[df_nc.treat == 1, 'y'].mean()
mean_assigned_c  = df_nc.loc[df_nc.treat == 0, 'y'].mean()
ITT              = mean_assigned_t - mean_assigned_c

print(f'Mean outcome — assigned treatment : {mean_assigned_t:.3%}')
print(f'Mean outcome — assigned control   : {mean_assigned_c:.3%}')
print(f'ITT estimate                      : {ITT:.3%}')
print()
print(f'For reference, clean RCT estimate : {lift:.3%}')

### 💬 Discussion

The ITT is smaller than the clean RCT estimate. Why?
The 30% non-compliers were assigned to treatment but received nothing,
so their outcome contribution pulls the treatment group mean down.

> *Your thoughts:*


### 9b. The wrong approach — comparing receivers vs non-receivers

What if you ignored the randomization and just compared visitors who
**received** the new page (`received=1`) to those who did not (`received=0`)?

This re-introduces selection bias: compliers (who received the page when assigned)
are systematically different from non-compliers. In real experiments,
non-compliers tend to be lower-engagement users — so the `received=1` group
is self-selected to be higher-intent, creating bias.


In [ ]:
mean_received_1  = df_nc.loc[df_nc.received == 1, 'y'].mean()
mean_received_0  = df_nc.loc[df_nc.received == 0, 'y'].mean()
naive_estimate   = mean_received_1 - mean_received_0

print(f'Mean outcome — received treatment : {mean_received_1:.3%}')
print(f'Mean outcome — did NOT receive    : {mean_received_0:.3%}')
print(f'Naive estimate                    : {naive_estimate:.3%}')
print()
print(f'ITT (assignment-based, unbiased)  : {ITT:.3%}')
print(f'Clean RCT estimate                : {lift:.3%}')

### 💬 Discussion

Is the naive estimate higher or lower than the ITT here?
In which direction would you generally expect the bias to go,
and why? (Think about who tends to be a non-complier in real experiments.)

> *Your thoughts:*


### 9c. Local Average Treatment Effect (LATE)

The LATE recovers the treatment effect for **compliers only** — visitors who
see the new page when assigned and don't when not assigned.

The formula uses the **Wald estimator**:

$$\text{LATE} = \frac{\text{ITT}}{\text{First Stage}}
= \frac{E[Y \mid Z=1] - E[Y \mid Z=0]}{E[D \mid Z=1] - E[D \mid Z=0]}$$

where $D$ = actual receipt indicator and $Z$ = assignment indicator.

The **first stage** is the compliance rate — the effect of assignment on receipt.
Since there are no always-takers in our setup, $E[D \mid Z=0] = 0$, so the
first stage simplifies to the fraction of assigned-treatment users who complied.


In [ ]:
# First stage: effect of ASSIGNMENT on actual RECEIPT
first_stage = (
    df_nc.loc[df_nc.treat == 1, 'received'].mean() -
    df_nc.loc[df_nc.treat == 0, 'received'].mean()
)

# LATE = ITT / first stage
LATE = ITT / first_stage

print(f'First stage (compliance rate) : {first_stage:.3%}')
print(f'ITT                           : {ITT:.3%}')
print(f'LATE  =  ITT / first_stage    : {LATE:.3%}')
print()
print('Full comparison:')
rows = [
    ('True effect (hidden)',        '3.000%'),
    ('Clean RCT (no non-compliance)', f'{lift:.3%}'),
    ('ITT (assignment-based)',      f'{ITT:.3%}'),
    ('LATE (compliers only)',       f'{LATE:.3%}'),
    ('Naive (biased)',              f'{naive_estimate:.3%}'),
]
for label, val in rows:
    print(f'  {label:<35} {val}')

### 💬 Discussion

1. **The LATE is larger than the ITT.** Will this always be true when compliance < 100%?
   Use the formula $\text{LATE} = \text{ITT} / \text{compliance rate}$ to explain.

2. **Which estimate should you report to your manager?**
   - If the decision is: *'Should we run this email campaign knowing some emails will bounce?'* → ITT or LATE?
   - If the decision is: *'Is the new page itself effective for people who actually see it?'* → ITT or LATE?

3. **Can we say anything about the effect for never-takers** (users who would never
   see the new page even if assigned)? Why or why not?

> *Your thoughts:*


### 9d. The Wald estimator (IV connection)

The LATE can also be derived as an **Instrumental Variables** estimate,
where assignment $Z$ is used as an instrument for actual receipt $D$.

$$\text{Wald} = \frac{\text{Reduced form}}{\text{First stage}}
= \frac{E[Y \mid Z=1] - E[Y \mid Z=0]}{E[D \mid Z=1] - E[D \mid Z=0]}$$

This is numerically identical to ITT / compliance rate.
The cell below verifies that.


In [ ]:
reduced_form = (
    df_nc.loc[df_nc.treat == 1, 'y'].mean() -
    df_nc.loc[df_nc.treat == 0, 'y'].mean()
)

wald = reduced_form / first_stage

print(f'Reduced form (= ITT)  : {reduced_form:.6f}')
print(f'First stage           : {first_stage:.6f}')
print(f'Wald estimator        : {wald:.6f}')
print(f'LATE from above       : {LATE:.6f}')
print(f'Are they identical?   : {abs(wald - LATE) < 1e-10}')

---
## ✏️ Before the reveal — your final guess

Based on everything we have seen:

- Our estimate was **8.75pp**
- The MDE for n=400 is **8.91pp** (we were technically underpowered)
- The sampling noise simulation showed estimates range from negative to very large

**What do you think the true lift actually was?**

> **My revised guess**: _____ percentage points

The instructor will reveal the answer by running the next cell.


In [ ]:
TRUE_LIFT = 0.03

print(f'True lift used to generate the data : {TRUE_LIFT:.1%}')
print(f'Our single-experiment estimate       : {lift:.2%}')
print(f'MDE for n = {N_PER_GROUP}                   : {approximate_mde(N_PER_GROUP):.2%}')
print()
n_needed = int(((stats.norm.ppf(0.975) + stats.norm.ppf(0.80)) * 0.45)**2 * 2 / TRUE_LIFT**2)
print(f'To reliably detect {TRUE_LIFT:.0%} at 80% power you need ~{n_needed} per group')
print(f'We used {N_PER_GROUP} per group — we were underpowered.')
print(f'We got a significant result anyway because our estimate was a lucky overestimate.')

### 💬 Final discussion

1. Our estimate (8.75pp) was much larger than the true effect (3pp).
   Our experiment was underpowered (MDE = 8.91pp > true lift = 3pp).
   Yet we got a significant result. How is that possible?

2. **Type M error** (magnitude error): when underpowered experiments produce
   significant results, they tend to dramatically over-estimate the true effect.
   If your manager acts on our 8.75pp estimate, what could go wrong?

3. What sample size would we have needed to reliably detect a 3pp effect?
   (Use the MDE table from Section 7.)

4. Looking back at your original prior from Section 1 — was it closer to
   our estimate (8.75pp) or the true lift (3pp)? What does that tell you
   about the value of prior information in experiment design?

> *Your thoughts:*
